# 05 - Model Comparison and Error Analysis

**Project:** Predictive Modeling for Drug Discovery via Virtual Screening  
**Student:** Milica Jeftic (ID: 89211255)  
**Date:** January 2026  
**Dataset:** Kaggle - Drug Discovery Virtual Screening Dataset

---

## Goal of This Notebook

This notebook combines the results from all modeling notebooks and provides the final model comparison and error analysis.

Models compared:

1. Logistic Regression baseline
2. Random Forest
3. Gradient Boosting
4. Neural Network

The goal is to identify the best-performing model, compare model families, inspect classification errors, and discuss the main limitation observed in this dataset.

---

## Expected Outputs

- Combined model comparison table
- Model comparison visualizations
- Best model identification
- Test-set error analysis
- Final interpretation and limitations
- Saved comparison metrics in `results/metrics/`


## 1. Environment Setup

This section imports libraries and defines project paths.

In [ ]:
# ============================
# Environment & Configuration
# ============================

import os
import sys
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import learning_curve, StratifiedKFold

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# ----------------------------
# Reproducibility & Warnings
# ----------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----------------------------
# Pandas display options
# ----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# ----------------------------
# Visualization defaults
# ----------------------------
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
rcParams["figure.figsize"] = (12, 6)
rcParams["font.size"] = 12

%matplotlib inline

# ----------------------------
# Project paths
# ----------------------------
def find_project_root(start_path):
    """Find the project root from either the repository root or the notebooks directory."""
    current_path = os.path.abspath(start_path)
    for _ in range(3):
        expected_items = [
            os.path.join(current_path, "data"),
            os.path.join(current_path, "notebooks"),
            os.path.join(current_path, "README.md"),
        ]
        if all(os.path.exists(path) for path in expected_items):
            return current_path
        current_path = os.path.dirname(current_path)
    raise FileNotFoundError("Could not locate the project root directory.")

PROJECT_ROOT = find_project_root(os.getcwd())
DATA_PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_PATH = os.path.join(PROJECT_ROOT, "models")
RESULTS_PATH = os.path.join(PROJECT_ROOT, "results")
METRICS_PATH = os.path.join(RESULTS_PATH, "metrics")
FIGURES_PATH = os.path.join(RESULTS_PATH, "figures")

print("=" * 60)
print("Environment initialized successfully")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"Numpy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print("-" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Metrics dir  : {METRICS_PATH}")
print(f"Models dir   : {MODELS_PATH}")
print("=" * 60)


## 2. Load Model Metrics

Metrics saved by notebooks 02, 03, and 04 are loaded and combined into one comparison table.

In [ ]:
print("=" * 60)
print("LOADING MODEL METRICS")
print("=" * 60)

metric_files = [
    "baseline_logistic_regression_metrics.csv",
    "tree_models_metrics.csv",
    "neural_network_metrics.csv",
]

metrics_tables = []
for file_name in metric_files:
    file_path = os.path.join(METRICS_PATH, file_name)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing metrics file: {file_path}")
    table = pd.read_csv(file_path)
    metrics_tables.append(table)
    print(f"Loaded: {file_name}")

all_metrics = pd.concat(metrics_tables, ignore_index=True)
metric_cols = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
all_metrics[metric_cols] = all_metrics[metric_cols].astype(float)

all_metrics_sorted = all_metrics.sort_values(["split", "roc_auc", "f1_score"], ascending=[True, False, False])
display(all_metrics_sorted)

combined_metrics_path = os.path.join(METRICS_PATH, "combined_model_metrics.csv")
all_metrics.to_csv(combined_metrics_path, index=False)
print(f"Combined metrics saved to: {combined_metrics_path}")


## 3. Validation and Test Comparison

Validation results are useful for model selection, while test results provide final held-out performance.

In [ ]:
validation_metrics = all_metrics[all_metrics["split"] == "validation"].copy()
test_metrics = all_metrics[all_metrics["split"] == "test"].copy()

print("Validation metrics:")
display(validation_metrics.sort_values(["roc_auc", "f1_score"], ascending=False))

print("Test metrics:")
display(test_metrics.sort_values(["roc_auc", "f1_score"], ascending=False))

best_validation_model = validation_metrics.sort_values(["roc_auc", "f1_score"], ascending=False).iloc[0]
best_test_model = test_metrics.sort_values(["roc_auc", "f1_score"], ascending=False).iloc[0]

print(f"Best validation model: {best_validation_model['model']}")
print(f"Best test model: {best_test_model['model']}")


## 4. Model Comparison Visualizations

The plots below compare F1-score and ROC-AUC across models on the test set.

In [ ]:
os.makedirs(FIGURES_PATH, exist_ok=True)

test_plot_df = test_metrics.melt(
    id_vars=["model", "split"],
    value_vars=["accuracy", "precision", "recall", "f1_score", "roc_auc"],
    var_name="metric",
    value_name="score",
)

plt.figure(figsize=(12, 6))
sns.barplot(data=test_plot_df, x="model", y="score", hue="metric")
plt.ylim(0.90, 1.01)
plt.title("Test Set Model Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
comparison_plot_path = os.path.join(FIGURES_PATH, "model_comparison_test_metrics.png")
plt.savefig(comparison_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Model comparison plot saved to: {comparison_plot_path}")


## 5. Load Test Data and Trained Models

The trained models are loaded from disk so their test-set errors can be analyzed consistently.

In [ ]:
print("=" * 60)
print("LOADING TEST DATA AND MODELS")
print("=" * 60)

X_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_test.csv"))
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_test.csv")).squeeze("columns").astype(int)

model_paths = {
    "Logistic Regression": os.path.join(MODELS_PATH, "baseline_logistic_regression.joblib"),
    "Random Forest": os.path.join(MODELS_PATH, "random_forest.joblib"),
    "Gradient Boosting": os.path.join(MODELS_PATH, "gradient_boosting.joblib"),
    "Neural Network": os.path.join(MODELS_PATH, "neural_network_mlp.joblib"),
}

models = {}
for model_name, model_path in model_paths.items():
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Missing model file for {model_name}: {model_path}")
    models[model_name] = joblib.load(model_path)
    print(f"Loaded {model_name}: {model_path}")

print(f"Test set shape: {X_test.shape}")
print(f"Test target distribution: {y_test.value_counts().sort_index().to_dict()}")


## 6. Test Error Analysis

For each model, the number of false positives and false negatives is calculated. In virtual screening, false negatives are especially important because they represent active compounds that would be missed.

In [ ]:
error_rows = []
error_examples = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = np.full(len(y_pred), np.nan)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    error_rows.append({
        "model": model_name,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "total_errors": fp + fn,
    })

    error_mask = y_test.to_numpy() != y_pred
    if error_mask.any():
        errors = X_test.loc[error_mask].copy()
        errors["true_label"] = y_test.to_numpy()[error_mask]
        errors["predicted_label"] = y_pred[error_mask]
        errors["predicted_probability_active"] = y_proba[error_mask]
        errors["model"] = model_name
        error_examples.append(errors)

error_summary = pd.DataFrame(error_rows)
display(error_summary)

error_summary_path = os.path.join(METRICS_PATH, "test_error_summary.csv")
error_summary.to_csv(error_summary_path, index=False)
print(f"Error summary saved to: {error_summary_path}")

if error_examples:
    error_examples_df = pd.concat(error_examples, ignore_index=True)
    error_examples_path = os.path.join(METRICS_PATH, "test_error_examples.csv")
    error_examples_df.to_csv(error_examples_path, index=False)
    print(f"Error examples saved to: {error_examples_path}")
    display(error_examples_df.head(10))
else:
    print("No test-set errors found for any loaded model.")


## 7. Feature Importance Review

The tree-based feature importance files are reviewed to support interpretation of why the tree models perform so strongly.

In [ ]:
rf_importance_path = os.path.join(METRICS_PATH, "random_forest_feature_importance.csv")
gb_importance_path = os.path.join(METRICS_PATH, "gradient_boosting_feature_importance.csv")

rf_importance = pd.read_csv(rf_importance_path)
gb_importance = pd.read_csv(gb_importance_path)

print("Top Random Forest features:")
display(rf_importance.head(10))

print("Top Gradient Boosting features:")
display(gb_importance.head(10))


## 8. Learning Curve Analysis

Learning curves are used to check how model performance changes as the amount of training data increases. For Logistic Regression, Random Forest, Gradient Boosting, and Neural Network, the curves below compare training F1-score with cross-validation F1-score.

For Logistic Regression and tree-based models, this is more appropriate than a loss trajectory because these models do not expose the same epoch-by-epoch loss history as a neural network. The Neural Network loss curve is already shown in notebook 04, while this section provides a consistent learning-curve comparison across all model families.


In [ ]:
print("=" * 60)
print("LEARNING CURVE ANALYSIS")
print("=" * 60)

X_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_train.csv"))
y_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_train.csv")).squeeze("columns").astype(int)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
train_sizes = np.linspace(0.2, 1.0, 5)

learning_curve_rows = []
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, (model_name, model) in zip(axes, models.items()):
    print(f"Computing learning curve for {model_name}...")
    train_sizes_abs, train_scores, cv_scores = learning_curve(
        estimator=model,
        X=X_train,
        y=y_train,
        train_sizes=train_sizes,
        cv=cv,
        scoring="f1",
        n_jobs=None,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    cv_mean = cv_scores.mean(axis=1)
    cv_std = cv_scores.std(axis=1)

    for size, tr_mean, tr_std, val_mean, val_std in zip(
        train_sizes_abs, train_mean, train_std, cv_mean, cv_std
    ):
        learning_curve_rows.append({
            "model": model_name,
            "train_size": int(size),
            "train_f1_mean": tr_mean,
            "train_f1_std": tr_std,
            "cv_f1_mean": val_mean,
            "cv_f1_std": val_std,
        })

    ax.plot(train_sizes_abs, train_mean, marker="o", label="Training F1")
    ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.15)
    ax.plot(train_sizes_abs, cv_mean, marker="s", label="CV F1")
    ax.fill_between(train_sizes_abs, cv_mean - cv_std, cv_mean + cv_std, alpha=0.15)
    ax.set_title(model_name)
    ax.set_xlabel("Training examples")
    ax.set_ylabel("F1-score")
    ax.set_ylim(0.85, 1.01)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Learning Curves by Model", fontsize=16, y=1.02)
plt.tight_layout()

learning_curve_path = os.path.join(FIGURES_PATH, "learning_curves_f1.png")
plt.savefig(learning_curve_path, dpi=300, bbox_inches="tight")
plt.show()

learning_curve_summary = pd.DataFrame(learning_curve_rows)
display(learning_curve_summary)

learning_curve_summary_path = os.path.join(METRICS_PATH, "learning_curve_summary.csv")
learning_curve_summary.to_csv(learning_curve_summary_path, index=False)

print(f"Learning curve figure saved to: {learning_curve_path}")
print(f"Learning curve summary saved to: {learning_curve_summary_path}")


### Learning Curve Interpretation

The learning curves show that all models reach high F1-scores even with smaller training subsets. This supports the earlier conclusion that the dataset is highly separable. The small gap between training and cross-validation scores suggests that the final models generalize well on this dataset, although this should still be interpreted carefully because the dataset is synthetic and strongly influenced by `binding_affinity`.

The learning-curve analysis is also a practical replacement for loss trajectories for Logistic Regression and tree-based models, since these model families do not naturally produce epoch-by-epoch loss curves like neural networks.


## 9. Final Model Comparison Summary

The final model comparison shows that all models perform very strongly on the processed virtual screening dataset.

### Main Findings

- Logistic Regression already provides a very strong baseline, with test F1-score of 0.9822 and test ROC-AUC of 0.9998.
- Random Forest and Gradient Boosting both achieve perfect validation and test scores on this dataset.
- The Neural Network also performs strongly, with test F1-score of 0.9762 and test ROC-AUC of 0.9993.
- The tree-based models are the best-performing models based on validation and test metrics.

### Error Analysis

The tree-based models make no errors on the test set. Logistic Regression makes three false positive errors and no false negatives, meaning it does not miss any active compounds. The Neural Network makes four total errors, including one false negative. High recall is important in virtual screening because missing active compounds can be more costly than allowing some false positives.

### Interpretation

The very high scores, especially the perfect tree-based model performance, suggest that the dataset is highly separable. Feature importance analysis shows that `binding_affinity` is the dominant predictor, especially for Gradient Boosting. This means the target variable is likely strongly related to binding affinity and related descriptors.

### Limitation

These results should be interpreted carefully. In real drug discovery workflows, binding affinity may not always be available before prediction. If the `active` label was derived from binding affinity, including it as a model input makes the classification task much easier and may overestimate real-world performance.

### Final Choice

For this dataset, Random Forest or Gradient Boosting can be selected as the best-performing final model. Random Forest is slightly more interpretable than Gradient Boosting in this analysis because its feature importance is distributed across several descriptors, while Gradient Boosting relies almost entirely on `binding_affinity`.
